In [ ]:
import os
import pandas as pd

# Caminho da pasta do OneDrive (ajuste conforme sua estrutura)
pasta = 

# Lista todos os arquivos XLSX da pasta
arquivos = [f for f in os.listdir(pasta) if f.endswith(".xlsx")]

# Dicionário de mapeamento TIPOMOVIMENTO → UNIDADE
mapa_unidade = {
    "CAVD": "Auto BB",
    "CAVE": "Auto BB",
    "UBMS": "Auto BB",
    "NAUTICO": "Empresas"
}

# Dicionário TIPOS DE SEGURO → TIPO DE DOCUMENTO
mapa_documento = {
    "7 - CARNE DE RESSARCIMENTO": "Carnê de Ressarcimento", 
    "8 - CARTA DE AVISO DE RENOVAÃ‡ÃƒO": "Carta de Aviso de Renovação", 
    "1 - ADESÃƒO - SEGURO NOVO": "Ap. e End.", 
    "550 - 05 Exclusão de Risco por Sinistro": "Ap. e End."
}

# Dicionário composto (TIPOMOVIMENTO + CAMPANHA) → PARCEIRO
# Chave = f"{TIPOMOVIMENTO}|{CAMPANHA}"
mapa_parceiro_composto = {
    "UBMS|0001-NÃO INFORMADO": "Brasil Veiculos",
    "UBMS|0010-ESTILO": "Brasil Veiculos",
    "VIMA|001-MAPFRE": "MAPFRE"
}

# Lista para armazenar dataframeMAPFRE
dfs = []
resumo_arquivos = []

for arquivo in arquivos:
    caminho = os.path.join(pasta, arquivo)
    df = pd.read_excel(caminho)
    linhas = len(df)

    # TIPOMOVIMENTO
    if "ARQUIVOMOVIMENTO" in df.columns:
        df["TIPOMOVIMENTO"] = df["ARQUIVOMOVIMENTO"].astype(str).str.split("_").str[0]
    else:
        df["TIPOMOVIMENTO"] = None

    # UNIDADE
    df["UNIDADE"] = df["TIPOMOVIMENTO"].map(mapa_unidade).fillna("Não Classificado")

    # PARCEIRO (com base em TIPOMOVIMENTO + CAMPANHA)
    if "CAMPANHA" in df.columns:
        # 🔧 AJUSTE: normaliza CAMPANHA para texto limpo com zeros à esquerda
        df["CAMPANHA"] = (
            df["CAMPANHA"]
            .astype(str)                     # converte tudo para texto
            .str.strip()                     # remove espaços
            .str.replace(r"\.0$", "", regex=True)  # remove .0 se veio de float
            .str.zfill(3)                    # garante formato 001, 1500, etc.
        )
        df["CHAVE_PARCEIRO"] = df["TIPOMOVIMENTO"].astype(str) + "|" + df["CAMPANHA"]
        df["PARCEIRO"] = df["CHAVE_PARCEIRO"].map(mapa_parceiro_composto).fillna("Não Classificado")
        df.drop(columns=["CHAVE_PARCEIRO"], inplace=True)
    else:
        df["PARCEIRO"] = df["TIPOMOVIMENTO"].map(lambda x: "Não Classificado")
        
    # IMPRESSO
    #if "NUMVIA" in df.columns:
        #df["IMPRESSO"] = df.apply(
            #lambda x: 1 if x.get("TIPOMOVIMENTO") == "UBMS" and pd.to_numeric(x.get("NUMVIA"), errors="coerce") >= 2 else 0,
            #axis=1
        #)
    #else:
        #df["IMPRESSO"] = 0

    # TIPO DE DOCUMENTO
    if "TIPOSEGURO" in df.columns:
        df["TIPO DE DOCUMENTO"] = df["TIPOSEGURO"].map(mapa_documento).fillna("Não Classificado")
    else:
        df["TIPO DE DOCUMENTO"] = "Não Classificado"

    dfs.append(df)
    resumo_arquivos.append({"Arquivo": arquivo, "Linhas": linhas})

# Junta todos
consolidado = pd.concat(dfs, ignore_index=True, sort=False)

# --- garantir colunas de valores existentes e numéricas (sem alterar dados originais) ---
for col in ["VALORPROCESSAMENTOEMAIL", "VALORCERTIFDIGITAL", "VALOREMAIL"]:
    if col not in consolidado.columns:
        consolidado[col] = 0
    consolidado[col] = pd.to_numeric(consolidado[col], errors="coerce").fillna(0)

# garantir TOTALENVIADOS e IMPRESSO como numéricos
if "TOTALENVIADOS" not in consolidado.columns:
    consolidado["TOTALENVIADOS"] = 0
consolidado["TOTALENVIADOS"] = pd.to_numeric(consolidado["TOTALENVIADOS"], errors="coerce").fillna(0)
#if "IMPRESSO" not in consolidado.columns:
    #consolidado["IMPRESSO"] = 0
#consolidado["IMPRESSO"] = pd.to_numeric(consolidado["IMPRESSO"], errors="coerce").fillna(0)

# Resumo agrupado (inclui PARCEIRO, PRODUTO, etc.)
group_cols = ["UNIDADE", "TIPOMOVIMENTO", "TIPOSEGURO", "TIPO DE DOCUMENTO", "PRODUTO", "PARCEIRO"]
if all(c in consolidado.columns for c in group_cols):
    resumo_group = (
        consolidado
        .groupby(group_cols, dropna=False)
        .agg(
            PROCESSADO=("TIPOMOVIMENTO", "count"),
            DIGITAL=("TOTALENVIADOS", lambda x: (pd.to_numeric(x, errors="coerce") == 0).sum()),  # TOTALENVIADOS=0
            EMAIL=("TOTALENVIADOS", lambda x: x[x >= 1].sum()),
            #IMPRESSO=("IMPRESSO", "sum"),
            # somas e médias auxiliares
            SUM_VALORPROCESSAMENTOEMAIL=("VALORPROCESSAMENTOEMAIL", "sum"),
            SUM_VALORCERTIFDIGITAL=("VALORCERTIFDIGITAL", "sum"),
            SUM_VALOREMAIL=("VALOREMAIL", "sum"),
            MEAN_VALORPROCESSAMENTOEMAIL=("VALORPROCESSAMENTOEMAIL", "mean"),
            MEAN_VALORCERTIFDIGITAL=("VALORCERTIFDIGITAL", "mean"),
            MEAN_VALOREMAIL=("VALOREMAIL", "mean"),
        )
        .reset_index()
    )

    # calcular os valores conforme solicitado (usando unitário médio * quantidade)
    resumo_group["VALORPROCESSADO"] = (
        (resumo_group["MEAN_VALORPROCESSAMENTOEMAIL"].fillna(0) + resumo_group["MEAN_VALORCERTIFDIGITAL"].fillna(0))
        * resumo_group["PROCESSADO"]
    )

    resumo_group["VALOREMAIL"] = (
        resumo_group["MEAN_VALOREMAIL"].fillna(0) * resumo_group["EMAIL"]
    )

    #resumo_group["VALORIMPRESSO"] = (
        #(resumo_group["MEAN_VALORPROCESSAMENTOEMAIL"].fillna(0) + resumo_group["MEAN_VALORCERTIFDIGITAL"].fillna(0))
        #* resumo_group["IMPRESSO"]
    #)

    # opcional: remover colunas auxiliares que não deseja no resumo final
    resumo_group = resumo_group.drop(columns=[
        "SUM_VALORPROCESSAMENTOEMAIL", "SUM_VALORCERTIFDIGITAL", "SUM_VALOREMAIL",
        "MEAN_VALORPROCESSAMENTOEMAIL", "MEAN_VALORCERTIFDIGITAL", "MEAN_VALOREMAIL"
    ])
else:
    resumo_group = pd.DataFrame()

# Salva Excel final
saida = os.path.join(pasta, "Consolidado.xlsx")
with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    consolidado.to_excel(writer, sheet_name="Consolidado", index=False)
    pd.DataFrame(resumo_arquivos).to_excel(writer, sheet_name="Resumo_Arquivos", index=False)
    if not resumo_group.empty:
        resumo_group.to_excel(writer, sheet_name="Resumo_Agrupado", index=False)

print(f"\n✅ Total de arquivos processados: {len(arquivos)}")
for r in resumo_arquivos:
    print(f"📄 {r['Arquivo']} → {r['Linhas']} linhas")
print("\n💾 Arquivo 'Consolidado.xlsx' gerado com sucesso!")
